In [ ]:
# ===== GA + ANN (MLP) with Geometric-Mean Fitness + SHAP-Guided Immigration (M=0.08) =====
# Author: Dr. Uysal (extended for classification + regression)

import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import random

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold   # <<< CHANGED
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    mean_absolute_percentage_error,   # <<< NEW
)
from sklearn.neural_network import MLPClassifier, MLPRegressor  # <<< CHANGED
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor  # <<< CHANGED

import shap  # pip install shap

# -----------------------------
# Load data
# -----------------------------
DF_PATH = r"bidnobidf.xlsx"  # <-- set your path
RANDOM_STATE = 40

df = pd.read_excel(DF_PATH, engine="openpyxl")
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# -----------------------------
# GA Parameters
# -----------------------------
POP_SIZE = 50
GENERATIONS = 100
MUTATION_RATE = 0.01
ALPHA = 0.85            # geometric mean weight on performance (1-ALPHA on sparsity)
ELITE_SIZE = 2
NUM_FEATURES = X.shape[1]
EPS = 1e-6              # for geometric mean stability
IMMIGRATION_FRAC = 0.08
rng = random.Random(RANDOM_STATE)

# -----------------------------
# Problem type helper
# -----------------------------
def ask_problem_type():
    """Ask user whether the problem is classification or regression."""
    while True:
        pt = input("Enter problem type ('classification' or 'regression'): ").strip().lower()
        if pt in ("classification", "regression"):
            return pt
        print("Invalid input. Please enter 'classification' or 'regression'.")

# -----------------------------
# Helpers: GA operators
# -----------------------------
def initialize_population(size, num_genes):
    """Random binary chromosomes with ≥2 ones."""
    pop = []
    while len(pop) < size:
        chrom = [rng.randint(0, 1) for _ in range(num_genes)]
        if sum(chrom) >= 2:
            pop.append(chrom)
    return pop

def tournament_selection(population, fitness_scores, k=3):
    """Pick best among k random individuals."""
    contestants = rng.sample(list(zip(population, fitness_scores)), k)
    return max(contestants, key=lambda t: t[1])[0]

def single_point_crossover(p1, p2):
    """Single-point crossover."""
    if len(p1) < 2:
        return p1[:], p2[:]
    point = rng.randint(1, len(p1) - 2)
    return p1[:point] + p2[point:], p2[:point] + p1[point:]

def mutate(chrom, rate=MUTATION_RATE):
    """Bit-flip mutation with ≥2-feature constraint."""
    out = [(g if rng.random() > rate else 1 - g) for g in chrom]
    if sum(out) < 2:
        # ensure at least 2 ones
        idxs = list(range(len(out)))
        rng.shuffle(idxs)
        for i in idxs[:2]:
            out[i] = 1
    return out

# -----------------------------
# SHAP-guided immigration utilities
# -----------------------------
def _mean_abs_shap_per_feature(shap_vals):
    """Return (n_features,) vector of mean |SHAP|; handle binary vs multiclass."""
    if isinstance(shap_vals, list):  # multiclass classification
        per_class = [np.mean(np.abs(sv), axis=0) for sv in shap_vals]
        return np.mean(np.vstack(per_class), axis=0)
    else:
        return np.mean(np.abs(shap_vals), axis=0)

def compute_shap_feature_probs(X_full, y_full, problem_type="classification", random_state=RANDOM_STATE):
    """
    i) Fit a fast surrogate (GBM)
    ii) Get SHAP values
    iii) Mean |SHAP| per feature
    iv) Normalize to sum to 1 -> probabilities for Bernoulli sampling
    """
    if problem_type == "classification":
        surrogate = GradientBoostingClassifier(
            n_estimators=200, learning_rate=0.1, max_depth=3,
            subsample=1.0, random_state=random_state
        )
    else:  # regression
        surrogate = GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.1, max_depth=3,
            subsample=1.0, random_state=random_state
        )

    surrogate.fit(X_full, y_full)
    explainer = shap.TreeExplainer(surrogate)
    shap_vals = explainer.shap_values(X_full)
    imp = _mean_abs_shap_per_feature(shap_vals)
    imp = np.maximum(imp, 0.0)
    total = imp.sum()
    if total <= 0:
        probs = np.ones(X_full.shape[1], dtype=float) / X_full.shape[1]
    else:
        probs = imp / total
    return probs  # length = n_features, sums to 1

def _sample_immigrant_from_probs(probs, min_ones=2):
    """Sample chromosome via Bernoulli(p_j); enforce >= min_ones by flipping top-p features."""
    chrom = [1 if rng.random() < p else 0 for p in probs]
    if sum(chrom) < min_ones:
        for j in np.argsort(probs)[::-1][:min_ones]:
            chrom[int(j)] = 1
    return chrom

def shap_guided_immigration(population, fitness_scores, probs):
    """
    Replace a constant fraction (IMMIGRATION_FRAC) of the worst individuals
    with SHAP-guided immigrants each generation.
    """
    if probs is None or len(population) == 0:
        return population

    M = max(1, int(round(POP_SIZE * IMMIGRATION_FRAC)))  # constant M (>=1)
    order = np.argsort(fitness_scores)  # ascending: worst first
    replace_idx = order[:M]

    pop_new = [ind[:] for ind in population]
    for idx in replace_idx:
        immigrant = _sample_immigrant_from_probs(probs, min_ones=2)
        pop_new[int(idx)] = immigrant

    return pop_new

# -----------------------------
# ANN pipelines (classification / regression)
# -----------------------------
def _ann_pipeline(problem_type="classification"):
    if problem_type == "classification":
        clf = MLPClassifier(
            hidden_layer_sizes=(8, 10),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=100,
            early_stopping=True,
            n_iter_no_change=15,
            random_state=RANDOM_STATE
        )
    else:  # regression
        clf = MLPRegressor(
            hidden_layer_sizes=(8, 10),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=300,          # a bit higher for regression if desired
            early_stopping=True,
            n_iter_no_change=15,
            random_state=RANDOM_STATE
        )

    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", clf)
    ])

# -----------------------------
# GA Fitness
# -----------------------------
def evaluate_fitness(chromosome, problem_type="classification"):
    """
    Return scalar fitness.
    - Classification: same as your original (Acc+EPS)^ALPHA * (Parsimony+EPS)^(1-ALPHA)
    - Regression: uses MAPE% as the ONLY error metric, transformed into a performance score.
    """
    k = sum(chromosome)
    if k < 2:
        return 0.0

    mask = np.array(chromosome, dtype=bool)
    X_sel = X.loc[:, mask]

    # Train-test split:
    if problem_type == "classification":
        # Stratify if labels are valid
        try:
            stratify_y = y if len(pd.Series(y).unique()) > 1 else None
        except Exception:
            stratify_y = None

        X_train, X_test, y_train, y_test = train_test_split(
            X_sel, y, test_size=0.2, random_state=RANDOM_STATE, stratify=stratify_y
        )
    else:  # regression: no stratification
        X_train, X_test, y_train, y_test = train_test_split(
            X_sel, y, test_size=0.2, random_state=RANDOM_STATE
        )

    model = _ann_pipeline(problem_type)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # sparsity reward (higher when fewer features are used)
    parsimony = 1.0 - (k / NUM_FEATURES)

    if problem_type == "classification":
        acc = accuracy_score(y_test, y_pred)
        f1w = f1_score(y_test, y_pred, average="weighted")

        # Geometric-mean objective (unchanged)
        fitness = (acc + EPS)**ALPHA * (parsimony + EPS)**(1 - ALPHA)

        print(f"Acc: {acc:.4f} | F1w: {f1w:.4f} | k={k:3d}/{NUM_FEATURES} | "
              f"Parsimony: {parsimony:.4f} | Fitness(geo): {fitness:.6f}")
    else:
        # Regression: error metric = MAPE% ONLY
        mape = mean_absolute_percentage_error(y_test, y_pred)  # 0–1
        mape_percent = mape * 100.0

        # Convert MAPE% to a performance-like score (higher is better)
        # e.g., perf = 1 / (1 + MAPE%)
        perf = 1.0 / (1.0 + mape_percent)

        # Geometric mean of performance and parsimony
        fitness = (perf + EPS)**ALPHA * (parsimony + EPS)**(1 - ALPHA)

        print(f"MAPE: {mape_percent:.4f}% | k={k:3d}/{NUM_FEATURES} | "
              f"Parsimony: {parsimony:.4f} | Fitness(geo-MAPE): {fitness:.6f}")

    return float(fitness)

def create_next_generation(
    population,
    fitness_scores,
    mutation_rate=MUTATION_RATE,
    elite_size=ELITE_SIZE,
    shap_probs=None,
    problem_type="classification"
):
    """Elitism + tournament + crossover + mutation, then SHAP-guided immigration."""
    # sort by fitness desc
    sorted_pop = [x for _, x in sorted(zip(fitness_scores, population), key=lambda p: p[0], reverse=True)]
    elites = sorted_pop[:elite_size]

    new_pop = elites.copy()
    while len(new_pop) < len(population):
        p1 = tournament_selection(population, fitness_scores)
        p2 = tournament_selection(population, fitness_scores)
        c1, c2 = single_point_crossover(p1, p2)
        new_pop.append(mutate(c1, mutation_rate))
        if len(new_pop) < len(population):
            new_pop.append(mutate(c2, mutation_rate))

    # quick re-eval before immigration
    new_fits = [evaluate_fitness(ch, problem_type) for ch in new_pop]

    # SHAP-guided immigration
    new_pop = shap_guided_immigration(new_pop, new_fits, shap_probs)

    return new_pop[:len(population)]

# -----------------------------
# Final Model: 5-fold CV + full retrain
# -----------------------------
def build_final_model_5fold(best_chromosome, problem_type="classification"):
    mask = np.array(best_chromosome, dtype=bool)
    selected_features = X.columns[mask]
    X_selected = X.loc[:, mask]

    print("\n=== Final Model (5-fold CV) with Best Chromosome ===")
    print(f"Selected {len(selected_features)}/{X.shape[1]} features")
    print("Selected feature names:")
    for f in selected_features:
        print(" -", f)

    if problem_type == "classification":
        splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        splits = splitter.split(X_selected, y)
    else:
        splitter = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        splits = splitter.split(X_selected, y)

    fold_perf_1, fold_perf_2 = [], []  # Acc & F1 for class, or MAPE & dummy for reg

    for fold_id, (tr_idx, te_idx) in enumerate(splits, start=1):
        X_train, X_test = X_selected.iloc[tr_idx], X_selected.iloc[te_idx]
        y_train, y_test = y.iloc[tr_idx], y.iloc[te_idx]

        model = _ann_pipeline(problem_type)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        print(f"\n[Fold {fold_id}]")
        if problem_type == "classification":
            acc = accuracy_score(y_test, y_pred)
            f1w = f1_score(y_test, y_pred, average="weighted")
            fold_perf_1.append(acc)
            fold_perf_2.append(f1w)

            print(f"Accuracy: {acc:.4f} | F1-weighted: {f1w:.4f}")
            print("Classification Report:")
            print(classification_report(y_test, y_pred, zero_division=0))
            print("Confusion Matrix:")
            print(confusion_matrix(y_test, y_pred))
        else:
            mape = mean_absolute_percentage_error(y_test, y_pred)
            mape_percent = mape * 100.0
            fold_perf_1.append(mape_percent)

            print(f"MAPE: {mape_percent:.4f}%")

    print("\n=== 5-fold CV Summary ===")
    if problem_type == "classification":
        print(f"Accuracy:     {float(np.mean(fold_perf_1)):.4f} ± {float(np.std(fold_perf_1)):.4f}")
        print(f"F1-weighted:  {float(np.mean(fold_perf_2)):.4f} ± {float(np.std(fold_perf_2)):.4f}")
    else:
        print(f"MAPE%:        {float(np.mean(fold_perf_1)):.4f}% ± {float(np.std(fold_perf_1)):.4f}%")

    final_model = _ann_pipeline(problem_type)
    final_model.fit(X_selected, y)
    print("\nFinal model trained on FULL selected data (ready for deployment).")

    clf = final_model.named_steps["clf"]
    print("\n=== ANN Hyperparameters ===")
    print(f"hidden_layer_sizes: {clf.hidden_layer_sizes}")
    print(f"activation: {clf.activation}")
    print(f"solver: {clf.solver}")
    print(f"alpha (L2): {clf.alpha}")
    print(f"learning_rate_init: {clf.learning_rate_init}")
    print(f"max_iter: {clf.max_iter}")
    # MLPClassifier & MLPRegressor both have these if early_stopping=True
    if hasattr(clf, "early_stopping"):
        print(f"early_stopping: {clf.early_stopping}")
    if hasattr(clf, "n_iter_no_change"):
        print(f"n_iter_no_change: {clf.n_iter_no_change}")

    return final_model

# -----------------------------
# GA Loop
# -----------------------------
def run_genetic_algorithm(problem_type="classification"):
    population = initialize_population(POP_SIZE, NUM_FEATURES)

    # Precompute SHAP-based per-feature probabilities once on the full dataset
    print("\nComputing SHAP feature importances for immigration probabilities...")
    shap_probs = compute_shap_feature_probs(X, y, problem_type=problem_type, random_state=RANDOM_STATE)
    print("Top 5 features by SHAP prob:",
          [(X.columns[i], float(p)) for i, p in sorted(enumerate(shap_probs), key=lambda t: t[1], reverse=True)[:5]])

    best_overall_fitness = -1.0
    best_overall_chromosome = None

    for gen in range(1, GENERATIONS + 1):
        print(f"\n--- Generation {gen} ---")
        fitness_scores = [evaluate_fitness(ch, problem_type) for ch in population]

        best_idx = int(np.argmax(fitness_scores))
        if fitness_scores[best_idx] > best_overall_fitness:
            best_overall_fitness = fitness_scores[best_idx]
            best_overall_chromosome = population[best_idx][:]

        print(f"Best (gen): fit={fitness_scores[best_idx]:.6f} | chrom={population[best_idx]}")

        population = create_next_generation(
            population,
            fitness_scores,
            mutation_rate=MUTATION_RATE,
            elite_size=ELITE_SIZE,
            shap_probs=shap_probs,
            problem_type=problem_type
        )

    print("\n=== Final Best Result ===")
    print(f"Best Fitness (geo): {best_overall_fitness:.6f}")
    print(f"Best Chromosome (1=selected): {best_overall_chromosome}")
    print(f"Selected features: {sum(best_overall_chromosome)}/{NUM_FEATURES}")

    if best_overall_chromosome is not None:
        _ = build_final_model_5fold(best_overall_chromosome, problem_type=problem_type)

# -----------------------------
# Execute
# -----------------------------
if __name__ == "__main__":
    problem_type = ask_problem_type()         # <<< NEW
    run_genetic_algorithm(problem_type)


